# EDA - Global Mobile Reviews Dataset

Analisis Exploratorio de Datos de resenas de telefonos moviles.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

from src.load_data import load_csv, summary_stats
from src.cleaning import adjust_dtypes, handle_missing, remove_duplicates
from src.analysis import descriptive_stats, correlation_analysis
from src.visualization import generate_all_visualizations

## 1. Carga de Datos

In [ ]:
df = load_csv('../data/dataset.csv')

In [ ]:
summary_stats(df)

## 2. Limpieza y Transformacion

In [ ]:
df = adjust_dtypes(df)
df = handle_missing(df, strategy='auto')
df = remove_duplicates(df)
df.info()

## 3. Analisis Descriptivo

In [ ]:
stats_df = descriptive_stats(df)

## 4. Deteccion de Outliers

In [ ]:
from src.cleaning import detect_outliers_iqr
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
outliers = detect_outliers_iqr(df, numeric_cols)
for col, info in outliers.items():
    if info['count'] > 0:
        print(f"{col}: {info['count']} outliers ({info['pct']}%)")

## 5. Correlacion de Variables

In [ ]:
corr = correlation_analysis(df)

In [ ]:
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, linewidths=0.5)
plt.title('Matriz de Correlacion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Visualizaciones

In [ ]:
generate_all_visualizations(df, corr)

## 7. Visualizaciones Adicionales en Notebook

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

brand_counts = df['brand'].value_counts()
axes[0,0].barh(brand_counts.index, brand_counts.values, color='steelblue')
axes[0,0].set_title('Resenas por Marca')

country_counts = df['country'].value_counts().head(10)
axes[0,1].bar(country_counts.index, country_counts.values, color='steelblue')
axes[0,1].set_title('Top 10 Paises')
axes[0,1].tick_params(axis='x', rotation=45)

ages = df['age']
axes[0,2].hist(ages, bins=30, edgecolor='white', color='steelblue')
axes[0,2].set_title('Distribucion de Edad')

ratings = df['rating']
axes[1,0].hist(ratings, bins=5, edgecolor='white', color='steelblue', align='left')
axes[1,0].set_title('Distribucion de Rating')
axes[1,0].set_xticks([1,2,3,4,5])

source_counts = df['source'].value_counts()
axes[1,1].pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%', startangle=90)
axes[1,1].set_title('Fuente de Resenas')

sns.boxplot(data=df, x='sentiment', y='price_usd', ax=axes[1,2], palette='Set2')
axes[1,2].set_title('Precio por Sentimiento')

plt.tight_layout()
plt.show()

In [ ]:
df_clean = df.copy()
df_clean.to_csv('../data/clean_data.csv', index=False)
print(f"Dataset limpio exportado: {df_clean.shape}")